# Association between Scale_1 and corrected cultural production

In [1]:
import sys

import duckdb
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

sys.path.insert(0, '..')
from config import ROOT, SESHAT_DB

UNSEEN_SPECIES = ROOT / 'data/processed/unseen_species.csv'
POLITIES_XLSX = ROOT / 'external/SeshatDatasetAnalysis/datasets/polities.xlsx'

pd.options.display.float_format = '{:.3f}'.format

## Data

In [2]:
unseen_species = pd.read_csv(UNSEEN_SPECIES)
print(len(unseen_species), 'home region and century rows')
unseen_species

273 home region and century rows


,home_seshat_region,century,f0,f0_min,f0_max,f1,f2,f>2
0,Afghanistan,1000,38.056,6.487,180.208,8,2,8
1,Afghanistan,1100,13.061,2.156,62.341,4,0,1
2,Afghanistan,1200,7.421,1.231,35.701,2,0,1
3,Afghanistan,1700,31.518,5.990,145.419,4,0,0
4,Anatolia-Caucasus,-500,20.154,2.262,139.630,3,0,0
...,...,...,...,...,...,...,...,...
268,Western Europe,1300,786.764,615.086,1003.724,158,16,47
269,Western Europe,1400,1155.694,924.650,1442.737,175,22,83
270,Western Europe,1500,5849.841,4837.371,7077.850,682,45,128
271,Western Europe,1600,24251.349,20520.233,28705.987,2203,90,288


In [3]:
scale_values = pd.read_excel(POLITIES_XLSX, sheet_name='SCV_Imputed', usecols=['PolityName', 'Year', 'Scale_1']).dropna()
print(len(scale_values), 'Scale_1 values for', scale_values['PolityName'].nunique(), 'polities')
scale_values

1152 Scale_1 values for 576 polities


,PolityName,Year,Scale_1
0,cn_qing_dyn_1,1644,8.070
1,cn_qing_dyn_1,1796,8.070
2,cn_qing_dyn_2,1796,8.188
3,cn_qing_dyn_2,1912,8.253
4,mx_basin_of_mexico_1,-6000,3.379
...,...,...,...
1475,gb_british_emp_2,1968,8.812
1496,de_empire_1,919,6.959
1497,de_empire_1,1125,6.959
1498,de_east_francia,842,6.338


In [4]:
con = duckdb.connect(str(SESHAT_DB), read_only=True)
polity_regions = con.sql('SELECT polity_new_id AS seshat_id, home_seshat_region FROM polity').df()
con.close()
polity_regions

,seshat_id,home_seshat_region
0,cn_qing_dyn_1,North China
1,cn_qing_dyn_2,North China
2,mx_basin_of_mexico_1,Mexico
3,mx_basin_of_mexico_2,Mexico
4,mx_basin_of_mexico_3,Mexico
...,...,...
859,fr_france_modern_2,Western Europe
860,eg_egypt_modern_1,Northeast Africa
861,eg_egypt_modern_2,Northeast Africa
862,sy_syria_modern,Levant


## Panel by home region and century

In [5]:
polity_scale = scale_values.groupby('PolityName').agg(
    start_year=('Year', 'min'),
    end_year=('Year', 'max'),
    complexity=('Scale_1', 'mean'),
).reset_index()
polity_scale = polity_scale.rename(columns={'PolityName': 'seshat_id'}).merge(polity_regions, on='seshat_id')
polity_scale

,seshat_id,start_year,end_year,complexity,home_seshat_region
0,Early Niynginya,1650,1897,4.312,East Africa
1,af_durrani_emp,1747,1826,6.393,Afghanistan
2,af_ghur_principality,1025,1215,6.097,Afghanistan
3,af_greco_bactrian_k,-256,-125,6.435,Afghanistan
4,af_hephthalite_emp,408,561,7.059,Afghanistan
...,...,...,...,...,...
571,ye_ziyad_dyn,822,1037,5.958,Arabia
572,zi_great_zimbabwe,1270,1550,5.263,Southern Africa
573,zi_mutapa,1450,1880,5.388,Southern Africa
574,zi_torwa_rozvi,1494,1850,5.614,Southern Africa


In [6]:
centuries = pd.DataFrame({'century': range(-500, 1800, 100)})
alive = polity_scale.merge(centuries, how='cross')
alive = alive[(alive['start_year'] <= alive['century'] + 99) & (alive['end_year'] >= alive['century'])]
alive

,seshat_id,start_year,end_year,complexity,home_seshat_region,century
21,Early Niynginya,1650,1897,4.312,East Africa,1600
22,Early Niynginya,1650,1897,4.312,East Africa,1700
45,af_durrani_emp,1747,1826,6.393,Afghanistan,1700
61,af_ghur_principality,1025,1215,6.097,Afghanistan,1000
62,af_ghur_principality,1025,1215,6.097,Afghanistan,1100
...,...,...,...,...,...,...
13238,zi_toutswe,700,1250,4.125,Southern Africa,800
13239,zi_toutswe,700,1250,4.125,Southern Africa,900
13240,zi_toutswe,700,1250,4.125,Southern Africa,1000
13241,zi_toutswe,700,1250,4.125,Southern Africa,1100


In [7]:
complexity = alive.groupby(['home_seshat_region', 'century'])['complexity'].mean().reset_index()
complexity

,home_seshat_region,century,complexity
0,Afghanistan,-300,6.435
1,Afghanistan,-200,5.711
2,Afghanistan,-100,4.987
3,Afghanistan,0,6.016
4,Afghanistan,100,7.045
...,...,...,...
613,Western North America,1100,5.111
614,Western North America,1200,5.111
615,Western North America,1300,5.111
616,Western North America,1400,5.111


In [8]:
production = unseen_species[['home_seshat_region', 'century']].copy()
production['cultural_production'] = unseen_species['f0'] + unseen_species['f1'] + unseen_species['f2'] + unseen_species['f>2']

panel = production.merge(complexity, on=['home_seshat_region', 'century'])
panel['log_cultural_production'] = np.log(panel['cultural_production'])
print(len(panel), 'home region and century rows |', panel['home_seshat_region'].nunique(), 'home regions')
panel

243 home region and century rows | 24 home regions


,home_seshat_region,century,cultural_production,complexity,log_cultural_production
0,Afghanistan,1000,56.056,6.097,4.026
1,Afghanistan,1100,18.061,6.097,2.894
2,Afghanistan,1200,10.421,6.097,2.344
3,Afghanistan,1700,35.518,6.393,3.570
4,Anatolia-Caucasus,-300,60.204,6.135,4.098
...,...,...,...,...,...
238,Western Europe,1300,1007.764,6.527,6.915
239,Western Europe,1400,1435.694,6.643,7.269
240,Western Europe,1500,6704.841,6.892,8.811
241,Western Europe,1600,26832.349,7.230,10.197


## Standardization

In [9]:
standardized = panel.copy()
for column in ['log_cultural_production', 'complexity']:
    standardized[column] = (panel[column] - panel[column].mean()) / panel[column].std()
standardized[['log_cultural_production', 'complexity']].describe()

,log_cultural_production,complexity
count,243.000,243.000
mean,0.000,-0.000
std,1.000,1.000
min,-1.889,-3.023
25%,-0.698,-0.630
50%,-0.011,-0.110
75%,0.550,0.641
max,3.265,2.528


## Model

In [10]:
MODELS = [
    ('log_cultural_production ~ complexity', None),
    ('log_cultural_production ~ complexity + C(century)', None),
    ('log_cultural_production ~ complexity', 'home_seshat_region'),
    ('log_cultural_production ~ complexity + C(century)', 'home_seshat_region'),
    ('complexity ~ log_cultural_production', None),
    ('complexity ~ log_cultural_production + C(century)', None),
    ('complexity ~ log_cultural_production', 'home_seshat_region'),
    ('complexity ~ log_cultural_production + C(century)', 'home_seshat_region'),
]

rows = []
for formula, random_intercept in MODELS:
    if random_intercept:
        model = smf.mixedlm(formula, standardized, groups=standardized[random_intercept]).fit(reml=False)
    else:
        model = smf.ols(formula, standardized).fit()
    predictor = formula.split(' ~ ')[1].split(' + ')[0]
    rows.append({
        'formula': formula,
        'random_intercept': random_intercept,
        'observations': int(model.nobs),
        'coefficient': model.params[predictor],
        'standard_error': model.bse[predictor],
        'p_value': model.pvalues[predictor],
        'aic': model.aic,
    })
results = pd.DataFrame(rows)
results['significant'] = results['p_value'] < 0.05
with pd.option_context('display.max_colwidth', None):
    display(results)

,formula,random_intercept,observations,coefficient,standard_error,p_value,aic,significant
0,log_cultural_production ~ complexity,NaN,243,0.396,0.059,0.000,651.053,True
1,log_cultural_production ~ complexity + C(century),NaN,243,0.372,0.056,0.000,607.620,True
2,log_cultural_production ~ complexity,home_seshat_region,243,0.436,0.065,0.000,596.204,True
3,log_cultural_production ~ complexity + C(century),home_seshat_region,243,0.367,0.051,0.000,467.935,True
4,complexity ~ log_cultural_production,NaN,243,0.396,0.059,0.000,651.053,True
5,complexity ~ log_cultural_production + C(century),NaN,243,0.453,0.068,0.000,655.448,True
6,complexity ~ log_cultural_production,home_seshat_region,243,0.351,0.052,0.000,539.950,True
7,complexity ~ log_cultural_production + C(century),home_seshat_region,243,0.475,0.066,0.000,530.168,True


## Model comparison

In [11]:
comparison = results[['formula', 'random_intercept', 'coefficient', 'p_value', 'significant', 'aic']].copy()
with pd.option_context('display.max_colwidth', None):
    display(comparison)

,formula,random_intercept,coefficient,p_value,significant,aic
0,log_cultural_production ~ complexity,NaN,0.396,0.000,True,651.053
1,log_cultural_production ~ complexity + C(century),NaN,0.372,0.000,True,607.620
2,log_cultural_production ~ complexity,home_seshat_region,0.436,0.000,True,596.204
3,log_cultural_production ~ complexity + C(century),home_seshat_region,0.367,0.000,True,467.935
4,complexity ~ log_cultural_production,NaN,0.396,0.000,True,651.053
5,complexity ~ log_cultural_production + C(century),NaN,0.453,0.000,True,655.448
6,complexity ~ log_cultural_production,home_seshat_region,0.351,0.000,True,539.950
7,complexity ~ log_cultural_production + C(century),home_seshat_region,0.475,0.000,True,530.168
